# 01: Data audit

Working notes for an evidence-based look at the Pump It Up data before cleaning, feature engineering or modelling.

I need to write the code, inspect the results and make the calls. These prompts are here to stop the audit becoming a tour of whichever columns happen to look interesting.

## Ground rule

Do not touch the source frames here. Count a problem before trying to fix it, record the evidence, then test the treatment in the baseline workflow.

Keep three statements separate for each feature:

- **Observation:** a fact supported by a count, summary or plot.
- **Interpretation:** my explanation of that fact.
- **Decision to test:** the treatment I plan to evaluate in the baseline.

Otherwise a hunch will become a cleaning rule, and I will forget why.

## Things I need to establish

1. Do the files have the expected structure and matching identifiers?
2. Which values represent genuine measurements, explicit nulls or hidden missing-data markers?
3. Does the target distribution create an imbalance problem?
4. Which features need transformation, grouping or exclusion?
5. Do related features describe the same underlying concept?
6. Does the test set differ from the training set in ways the preprocessing must handle?

## 1. Set up the notebook

Import the libraries needed for tabular analysis and plotting. Keep paths relative to the project; nobody else has my directory layout.

Expected local files:

- `TrainingSetValues.csv`
- `TrainingSetLabels.csv`
- `TestSetValues.csv`
- `SubmissionFormat.csv`

In [1]:
# Imports and project-relative data paths go here.
# Check that all four files exist before carrying on.

# Imports
print("Importing libraries...")
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
print("Imports completed")

# Data file presence checks
print("Confirming presence of data files...")
stage_directory = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "data" / "TrainingSetValues.csv").is_file()
)
training_set_values_path = stage_directory / "data" / "TrainingSetValues.csv"
training_set_labels_path = stage_directory / "data" / "TrainingSetLabels.csv"
test_set_values_path = stage_directory / "data" / "TestSetValues.csv"
submission_format_path = stage_directory / "data" / "SubmissionFormat.csv"

loadPaths = [
    training_set_values_path,
    training_set_labels_path,
    test_set_values_path,
    submission_format_path
]

for path in loadPaths:
    if not path.is_file():
        raise FileNotFoundError(f"Required file not found: {path}")

    print(f"{path} - file found OK")

print("Data file checks complete")



Importing libraries...


Imports completed
Confirming presence of data files...
C:\_Source\Imperial-ML-AI-Live\Capstone\imperial-capstone\stage-1-pump-it-up\data\TrainingSetValues.csv - file found OK
C:\_Source\Imperial-ML-AI-Live\Capstone\imperial-capstone\stage-1-pump-it-up\data\TrainingSetLabels.csv - file found OK
C:\_Source\Imperial-ML-AI-Live\Capstone\imperial-capstone\stage-1-pump-it-up\data\TestSetValues.csv - file found OK
C:\_Source\Imperial-ML-AI-Live\Capstone\imperial-capstone\stage-1-pump-it-up\data\SubmissionFormat.csv - file found OK
Data file checks complete


## 2. Load the four source files

One DataFrame per file. Keep the training values and labels separate until the identifier check passes.

Pick dull, consistent variable names. Future me will cope.

In [2]:
# Preserve source blanks and literal strings such as "None" separately.
training_set_values = pd.read_csv(
    training_set_values_path,
    keep_default_na=False,
)
training_set_labels = pd.read_csv(
    training_set_labels_path,
    keep_default_na=False,
)
test_set_values = pd.read_csv(
    test_set_values_path,
    keep_default_na=False,
)
submission_format = pd.read_csv(
    submission_format_path,
    keep_default_na=False,
)

print("Data loaded ok")
print("==============")
print("training_set_values:", training_set_values.shape[0])
print("training_set_labels:", training_set_labels.shape[0])
print("test_set_values:", test_set_values.shape[0])
print("submission_format:", submission_format.shape[0])


Data loaded ok
training_set_values: 59400
training_set_labels: 59400
test_set_values: 14850
submission_format: 14850


## 3. Check file structure and alignment

Checks to write for each DataFrame:

- row and column counts;
- column names and data types;
- duplicate identifiers;
- whether training IDs match label IDs;
- whether test IDs match the submission template;
- whether train and test contain the same predictor columns.

A failed alignment check stops the audit. There is little value analysing rows that may not belong together.

In [3]:
# Import the shared source-data validators instead of defining them here.
import sys

source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)


In [4]:
# Validate each source frame and the two ID pairings.
print(">>> Validating training feature schema...")
validate_raw_feature_schema(training_set_values)
print("Training feature schema is valid.")

print()
print(">>> Validating test feature schema...")
validate_raw_feature_schema(test_set_values)
print("Test feature schema is valid.")

print()
print(">>> Validating training labels...")
validate_label_frame(training_set_labels)
print("Training labels are valid.")

print()
print(">>> Validating submission format...")
validate_label_frame(submission_format)
print("Submission format is valid.")

print()
print(">>> Matching training feature and label IDs...")
validate_aligned_ids(training_set_values, training_set_labels)
print("Training feature and label IDs match in row order.")

print()
print(">>> Matching test feature and submission IDs...")
validate_aligned_ids(test_set_values, submission_format)
print("Test feature and submission IDs match in row order.")


>>> Validating training feature schema...
Training feature schema is valid.

>>> Validating test feature schema...
Test feature schema is valid.

>>> Validating training labels...
Training labels are valid.

>>> Validating submission format...
Submission format is valid.

>>> Matching training feature and label IDs...
Training feature and label IDs match in row order.

>>> Matching test feature and submission IDs...
Test feature and submission IDs match in row order.


## 4. Take a first look

Start with a small sample, then produce a column-level summary containing:

- data type;
- non-null and null counts;
- percentage missing;
- number of distinct values;
- one or two example values.

A transposed sample may be less unpleasant than scrolling through forty columns.

In [5]:
# Display a small sample and an aligned column-level schema summary.
display(training_set_values.head().T)

column_summary = pd.DataFrame({
    "dtype": training_set_values.dtypes.astype("string"),
    "example": training_set_values.iloc[0],
    "source blank rows": [
        int(training_set_values[column].astype("string").str.strip().eq("").sum())
        for column in training_set_values.columns
    ],
    "unique values": training_set_values.nunique(dropna=False),
})
display(column_summary)


,0,1,2,3,4
id,69572,8776,34310,67743,19728
amount_tsh,6000.0,0.0,25.0,0.0,0.0
date_recorded,2011-03-14,2013-03-06,2013-02-25,2013-01-28,2011-07-13
funder,Roman,Grumeti,Lottery Club,Unicef,Action In A
gps_height,1390,1399,686,263,0
installer,Roman,GRUMETI,World vision,UNICEF,Artisan
longitude,34.938093,34.698766,37.460664,38.486161,31.130847
latitude,-9.856322,-2.147466,-3.821329,-11.155298,-1.825359
wpt_name,none,Zahanati,Kwa Mahundi,Zahanati Ya Nanyumbu,Shuleni
num_private,0,0,0,0,0


,dtype,example,source blank rows,unique values
id,int64,69572,0,59400
amount_tsh,float64,6000.0,0,98
date_recorded,str,2011-03-14,0,356
funder,str,Roman,3635,1898
gps_height,int64,1390,0,2428
installer,str,Roman,3655,2146
longitude,float64,34.938093,0,57516
latitude,float64,-9.856322,0,57517
wpt_name,str,none,0,37400
num_private,int64,0,0,65


# Per-Column Initial Examination

I think before proceeding, I can manualy examine the column summaries, and try to establish or otherwise figure out for myself what they signify so that they may be processed properly.

## id

id is only used to cross-reference X and y. It is not a feature and should be removed from X before any model training.

## amount_tsh
### amount_tsh - Data Type: float64

**Difficult therefore potentially useful!**

The DrivenData page appears to have incorrectly expanded amount_tsh as total static head.

The documentation for the underlying Tanzanian Water Point Mapping data defines AMOUNT_TSH as the amount of payment received for water use.

When I deal with this data I will assume payment received. 

### amount_tsh - Observations
amount_tsh contains a large concentration of zero values and an extremely long upper tail. Among positive observations, the highest 1% account for much of the extreme skewness and kurtosis, although the remaining values are still substantially right-skewed and heavy-tailed.

Most points track the red line of the Q-Q plot reasonably well, especially through the centre. That means log10(amount_tsh) is much closer to a normal distribution than amount_tsh itself.

Positive amount_tsh is approximately log-normal in its broad shape, but strongly heaped at repeated round monetary values and imperfect in the tails.

For modelling, a log transformation is therefore quite defensible:

    training_set_values["amount_tsh_log"] = np.log1p(
        training_set_values["amount_tsh"]
    )

log1p also handles zeroes, though I would retain a separate zero/positive indicator:

    training_set_values["amount_tsh_is_positive"] = (
        training_set_values["amount_tsh"] > 0
    )
### amount_tsh - Relationship to the target

A positive `amount_tsh` is strongly associated with better waterpoint
status. Among observations with an amount of zero, 47.3% are functional
and 45.4% are non-functional. Among observations with a positive amount,
70.7% are functional and 22.2% are non-functional.

Higher positive amounts are also broadly associated with an increasing
functional share, particularly above approximately 500–1,000 TSh.
However, this relationship is not strictly monotonic across the lower
amount bands, and the highest bands contain fewer observations.

This is an association rather than evidence that charging or receiving
more money causes a waterpoint to function. `amount_tsh` may also proxy
for payment policy, management quality, population served, location or
whether a functioning waterpoint is capable of collecting payments.

For modelling, retain two derived signals:

- whether `amount_tsh` is positive;
- `log1p(amount_tsh)`, to preserve magnitude while compressing its
  extreme right tail.

The feature should later be considered alongside `payment`,
`payment_type`, region and other potentially related variables.

### amount_tsh - Later Investigation
My LLM had some thoughts on this, including that some data points might be the aggregation of several nearby smaller datapoints, which themselves then get recorded as zero. 

I wonder if I could take this data, plus coordinate data, to generate a heatmap. Does the data give a way to differentiate 'aggregator' entries from non-aggregators? Is that even what's happening?

Revisit amount_tsh later during the deep relationship-to-target pass, particularly against:

* status_group
* payment
* payment_type
* region
* basin



## 5. Audit the target

Count each `status_group` class, calculate its share of the training rows and add one readable chart.

Things to pin down:

- the majority-class accuracy a trivial classifier would achieve;
- whether accuracy alone would hide poor performance on a smaller class;
- which additional metric from the course would expose that weakness;
- whether later resampling must happen after the validation split.

In [6]:
# Calculate and plot the target distribution.
# Record the majority-class baseline and my metric choice.


## 6. Create a feature register

Work through the predictors in groups. I want one row per feature with these fields:

| Field | Purpose |
| --- | --- |
| Feature | Column name |
| Meaning | My plain-English interpretation |
| Review type | Numeric, categorical, binary, date, identifier/text or paired |
| Quality finding | Missingness, sentinels, range or cardinality issue |
| Relationship | Parent, child, duplicate concept or geographic pair |
| Target observation | Evidence of a relationship with `status_group` |
| Baseline decision to test | Keep, exclude, impute, group or derive |
| Confidence | High, medium or low, with a reason |

A DataFrame will help with sorting. A short Markdown table may survive better in the final write-up.

In [7]:
feature_register = pd.DataFrame([{'feature': 'amount_tsh', 'review type': 'numeric', 'focused audit': '../../01-amount_tsh/04-amount_tsh-deep-dive.ipynb', 'audit status': 'complete', 'key finding': 'Zero dominates and positive values are strongly right-skewed.', 'baseline treatment': 'Keep amount availability and compare transformed positive values.'}, {'feature': 'longitude + latitude', 'review type': 'paired geographic', 'focused audit': '20-longitude-latitude-pair-deep-dive.ipynb', 'audit status': 'complete', 'key finding': 'The pair (0, approximately 0) encodes 1,812 missing training locations.', 'baseline treatment': 'Keep availability; replace the paired sentinel and engineer location jointly.'}, {'feature': 'gps_height', 'review type': 'numeric', 'focused audit': '../feature-family-audits/20-numeric-feature-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'Zero affects 34.41% of training rows; 1,496 negative heights are stable in test.', 'baseline treatment': 'Flag zero, impute it inside each fold, and preserve negative values initially.'}, {'feature': 'num_private', 'review type': 'numeric', 'focused audit': '../feature-family-audits/20-numeric-feature-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'The field is undocumented and 98.73% zero; only 757 training rows are positive.', 'baseline treatment': 'Keep a non-zero flag plus magnitude candidate, with an early omission ablation.'}, {'feature': 'population', 'review type': 'numeric', 'focused audit': '../feature-family-audits/20-numeric-feature-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'Zero affects 36.00%; value one is a separate 11.83% spike with 53.52% non-functional.', 'baseline treatment': 'Keep zero and one flags plus log1p magnitude; do not merge the two special states.'}, {'feature': 'date_recorded', 'review type': 'temporal', 'focused audit': '../feature-family-audits/21-temporal-feature-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'All dates parse; training spans 2002-10-14 to 2013-12-03 with strong survey-month differences.', 'baseline treatment': 'Derive recording year/month and elapsed days; do not one-hot the raw date string.'}, {'feature': 'construction_year', 'review type': 'temporal', 'focused audit': '../feature-family-audits/21-temporal-feature-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'Year zero affects 34.86%; valid pump age has median 13 and nine negative training cases.', 'baseline treatment': 'Keep unknown/inconsistent flags and valid pump age; compare construction cohort.'}, {'feature': 'public_meeting', 'review type': 'binary', 'focused audit': '../feature-family-audits/22-binary-feature-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'The source is blank for 3,334 training rows (5.61%) and 821 test rows.', 'baseline treatment': 'Retain true/false/unknown as three explicit states.'}, {'feature': 'permit', 'review type': 'binary', 'focused audit': '../feature-family-audits/22-binary-feature-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'The source is blank for 3,056 training rows (5.15%) and 737 test rows.', 'baseline treatment': 'Retain true/false/unknown as three explicit states.'}, {'feature': 'basin', 'review type': 'geographic categorical', 'focused audit': '../feature-family-audits/23-geographic-category-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'Nine complete levels; train/test total-variation distance is 1.23%.', 'baseline treatment': 'Retain as a stable low-cardinality category.'}, {'feature': 'subvillage', 'review type': 'geographic categorical', 'focused audit': '../feature-family-audits/23-geographic-category-family-audit.ipynb', 'audit status': 'complete', 'key finding': '19,287 levels; 87.32% of rows are in <50 groups and 16.09% of test rows are unseen.', 'baseline treatment': 'Exclude from first one-hot baseline; validate hashing/frequency treatment separately.'}, {'feature': 'region', 'review type': 'geographic categorical', 'focused audit': '../feature-family-audits/23-geographic-category-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'Twenty-one complete levels; functional rates range from 29.75% to 78.22%.', 'baseline treatment': 'Retain as an interpretable low-cardinality back-off.'}, {'feature': 'region_code', 'review type': 'geographic categorical', 'focused audit': '../feature-family-audits/23-geographic-category-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'Twenty-seven categorical codes; region and code are not simple duplicates.', 'baseline treatment': 'Cast to category and compare with named region.'}, {'feature': 'district_code', 'review type': 'geographic categorical', 'focused audit': '../feature-family-audits/23-geographic-category-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'Twenty reused numeric labels; code zero is not universally missing.', 'baseline treatment': 'Cast to category and combine with region_code if used.'}, {'feature': 'lga', 'review type': 'geographic categorical', 'focused audit': '../feature-family-audits/23-geographic-category-family-audit.ipynb', 'audit status': 'complete', 'key finding': '125 complete levels, no unseen test levels, and deterministic mapping to region.', 'baseline treatment': 'Retain; compare its signal against region back-off.'}, {'feature': 'ward', 'review type': 'geographic categorical', 'focused audit': '../feature-family-audits/23-geographic-category-family-audit.ipynb', 'audit status': 'complete', 'key finding': '2,092 levels; raw unseen exposure is 0.07%, or 0.08% for LGA+ward.', 'baseline treatment': 'Retain via LGA+ward with fold-fitted rare/unseen handling.'}, {'feature': 'funder', 'review type': 'high-cardinality categorical', 'focused audit': '../feature-family-audits/24-organisation-and-name-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'Effective missingness is 7.48%; 1.71% of test rows have unseen funders.', 'baseline treatment': 'Keep null/sentinel distinct, normalise conservatively, then rare-pool inside folds.'}, {'feature': 'installer', 'review type': 'high-cardinality categorical', 'focused audit': '../feature-family-audits/24-organisation-and-name-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'Effective missingness is 7.54%; case normalisation reduces 2,145 to 1,918 levels.', 'baseline treatment': 'Keep separately from funder; trim/casefold/whitespace-normalise and rare-pool.'}, {'feature': 'wpt_name', 'review type': 'high-cardinality categorical', 'focused audit': '../feature-family-audits/24-organisation-and-name-family-audit.ipynb', 'audit status': 'complete', 'key finding': '56.90% of test names are unseen; leave-one-out lookup accuracy is only 55.01%.', 'baseline treatment': 'Exclude raw exact names from baseline; test cross-fitted text/frequency features later.'}, {'feature': 'scheme_management', 'review type': 'related categorical', 'focused audit': '../feature-family-audits/25-management-feature-family-audit.ipynb', 'audit status': 'complete', 'key finding': '3,877 training rows are blank; it overlaps management but is not equivalent.', 'baseline treatment': 'Retain unknown as a level; compare against management by ablation.'}, {'feature': 'scheme_name', 'review type': 'related categorical', 'focused audit': '../feature-family-audits/25-management-feature-family-audit.ipynb', 'audit status': 'complete', 'key finding': '28,166 rows are blank, 644 literal None; 1.26% of test rows use unseen names.', 'baseline treatment': 'Omit raw one-hot form initially; preserve blank/None and test fold-fitted pooling later.'}, {'feature': 'management', 'review type': 'related categorical', 'focused audit': '../feature-family-audits/25-management-feature-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'Twelve complete levels map deterministically to five management_group levels.', 'baseline treatment': 'Use as the initial granular management representation.'}, {'feature': 'management_group', 'review type': 'related categorical', 'focused audit': '../feature-family-audits/25-management-feature-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'Deterministic coarse parent retains much less descriptive target association.', 'baseline treatment': 'Treat as likely redundant; compare coarse versus granular representation.'}, {'feature': 'extraction_type', 'review type': 'categorical hierarchy', 'focused audit': '../feature-family-audits/26-infrastructure-feature-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'Eighteen levels map deterministically upward; `other` is 80.79% non-functional.', 'baseline treatment': 'Keep the granular type; rare-pool if validation requires it.'}, {'feature': 'extraction_type_group', 'review type': 'categorical hierarchy', 'focused audit': '../feature-family-audits/26-infrastructure-feature-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'Deterministic from type and only 0.0013 descriptive MI bits lower.', 'baseline treatment': 'Treat as a strong redundancy candidate beside extraction_type.'}, {'feature': 'extraction_type_class', 'review type': 'categorical hierarchy', 'focused audit': '../feature-family-audits/26-infrastructure-feature-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'Seven deterministic broad classes give a compact back-off.', 'baseline treatment': 'Compare type plus class against type alone; do not keep all three.'}, {'feature': 'source', 'review type': 'categorical hierarchy', 'focused audit': '../feature-family-audits/26-infrastructure-feature-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'Ten fully test-covered levels retain lake/river differences hidden by source_type.', 'baseline treatment': 'Keep granular source and compare with the intermediate type.'}, {'feature': 'source_type', 'review type': 'categorical hierarchy', 'focused audit': '../feature-family-audits/26-infrastructure-feature-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'Deterministic from source and removes useful within-type detail.', 'baseline treatment': 'Use only as a lower-cardinality ablation candidate.'}, {'feature': 'source_class', 'review type': 'categorical hierarchy', 'focused audit': '../feature-family-audits/26-infrastructure-feature-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'Three deterministic classes have weak descriptive association.', 'baseline treatment': 'First source-hierarchy level to omit from baseline.'}, {'feature': 'payment_type', 'review type': 'low-cardinality categorical', 'focused audit': '../feature-family-audits/27-service-category-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'Seven test-covered levels; unknown is 51.45% non-functional versus 38.42% overall.', 'baseline treatment': 'Retain unknown/other explicitly after removing duplicate payment.'}, {'feature': 'water_quality', 'review type': 'low-cardinality categorical', 'focused audit': '../feature-family-audits/27-service-category-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'Eight levels map deterministically upward; unknown is 84.06% non-functional.', 'baseline treatment': 'Keep granular quality and rare-pool the 17-row fluoride-abandoned level.'}, {'feature': 'quality_group', 'review type': 'low-cardinality categorical', 'focused audit': '../feature-family-audits/27-service-category-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'Deterministic parent hides the salty versus salty-abandoned repair difference.', 'baseline treatment': 'Treat as likely redundant beside water_quality.'}, {'feature': 'quantity', 'review type': 'low-cardinality categorical', 'focused audit': '../feature-family-audits/27-service-category-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'Dry covers 6,246 rows and is 96.89% non-functional; quantity_group is its duplicate.', 'baseline treatment': 'Retain as nominal; preserve unknown and never reintroduce quantity_group.'}, {'feature': 'waterpoint_type', 'review type': 'low-cardinality categorical', 'focused audit': '../feature-family-audits/27-service-category-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'Seven levels; `other` is 82.24% non-functional and `dam` has only seven rows.', 'baseline treatment': 'Keep granular type with infrequent handling.'}, {'feature': 'waterpoint_type_group', 'review type': 'low-cardinality categorical', 'focused audit': '../feature-family-audits/27-service-category-family-audit.ipynb', 'audit status': 'complete', 'key finding': 'Deterministic parent hides single-versus-multiple standpipe differences.', 'baseline treatment': 'Treat as likely redundant; use only as coarse ablation.'}])
assert len(feature_register) == 35  # 36 candidate columns; coordinates are one paired row.
assert feature_register["audit status"].eq("complete").all()
display(feature_register)


,feature,review type,focused audit,audit status,key finding,baseline treatment
0,amount_tsh,numeric,../../01-amount_tsh/04-amount_tsh-deep-dive.ipynb,complete,Zero dominates and positive values are strongl...,Keep amount availability and compare transform...
1,longitude + latitude,paired geographic,20-longitude-latitude-pair-deep-dive.ipynb,complete,"The pair (0, approximately 0) encodes 1,812 mi...",Keep availability; replace the paired sentinel...
2,gps_height,numeric,../feature-family-audits/20-numeric-feature-fa...,complete,"Zero affects 34.41% of training rows; 1,496 ne...","Flag zero, impute it inside each fold, and pre..."
3,num_private,numeric,../feature-family-audits/20-numeric-feature-fa...,complete,The field is undocumented and 98.73% zero; onl...,"Keep a non-zero flag plus magnitude candidate,..."
4,population,numeric,../feature-family-audits/20-numeric-feature-fa...,complete,Zero affects 36.00%; value one is a separate 1...,Keep zero and one flags plus log1p magnitude; ...
5,date_recorded,temporal,../feature-family-audits/21-temporal-feature-f...,complete,All dates parse; training spans 2002-10-14 to ...,Derive recording year/month and elapsed days; ...
6,construction_year,temporal,../feature-family-audits/21-temporal-feature-f...,complete,Year zero affects 34.86%; valid pump age has m...,Keep unknown/inconsistent flags and valid pump...
7,public_meeting,binary,../feature-family-audits/22-binary-feature-fam...,complete,"The source is blank for 3,334 training rows (5...",Retain true/false/unknown as three explicit st...
8,permit,binary,../feature-family-audits/22-binary-feature-fam...,complete,"The source is blank for 3,056 training rows (5...",Retain true/false/unknown as three explicit st...
9,basin,geographic categorical,../feature-family-audits/23-geographic-categor...,complete,Nine complete levels; train/test total-variati...,Retain as a stable low-cardinality category.


### Focused review order and coverage

The remaining audit is organised by analytical treatment rather than CSV order:

1. [`amount_tsh`](../../01-amount_tsh/04-amount_tsh-deep-dive.ipynb) and the paired [`longitude` / `latitude`](20-longitude-latitude-pair-deep-dive.ipynb) audits.
2. [Numeric magnitude](../feature-family-audits/20-numeric-feature-family-audit.ipynb), [temporal fields](../feature-family-audits/21-temporal-feature-family-audit.ipynb) and [nullable binary fields](../feature-family-audits/22-binary-feature-family-audit.ipynb).
3. [Geographic categories](../feature-family-audits/23-geographic-category-family-audit.ipynb) and [high-cardinality organisations/names](../feature-family-audits/24-organisation-and-name-family-audit.ipynb).
4. Related [management](../feature-family-audits/25-management-feature-family-audit.ipynb), [extraction/source](../feature-family-audits/26-infrastructure-feature-family-audit.ipynb) and [service](../feature-family-audits/27-service-category-family-audit.ipynb) categories.

Together these notebooks cover all 36 candidate predictors after the fixed structural removal, counting longitude and latitude separately. The templates below remain the governing checklist; the focused notebooks contain their executed evidence.


## 7. Numeric feature template

Use the same routine for each numeric feature:

1. report count, missingness, distinct values, minimum, quartiles, maximum and mean;
2. count zeros and decide whether zero can represent a real measurement;
3. inspect a histogram and a box plot;
4. compare the distribution across target classes;
5. note skew, outliers and implausible ranges;
6. record a treatment to test. Do not change the column here.

Begin with `amount_tsh`, get the routine into decent shape, then reuse it. Zero is not missing merely because it is inconvenient.

In [8]:
display(feature_register.loc[feature_register["review type"].eq("numeric")])
print("Executed distribution, sentinel, target-state and drift evidence: ../feature-family-audits/20-numeric-feature-family-audit.ipynb")


,feature,review type,focused audit,audit status,key finding,baseline treatment
0,amount_tsh,numeric,../../01-amount_tsh/04-amount_tsh-deep-dive.ipynb,complete,Zero dominates and positive values are strongl...,Keep amount availability and compare transform...
2,gps_height,numeric,../feature-family-audits/20-numeric-feature-fa...,complete,"Zero affects 34.41% of training rows; 1,496 ne...","Flag zero, impute it inside each fold, and pre..."
3,num_private,numeric,../feature-family-audits/20-numeric-feature-fa...,complete,The field is undocumented and 98.73% zero; onl...,"Keep a non-zero flag plus magnitude candidate,..."
4,population,numeric,../feature-family-audits/20-numeric-feature-fa...,complete,Zero affects 36.00%; value one is a separate 1...,Keep zero and one flags plus log1p magnitude; ...


Executed distribution, sentinel, target-state and drift evidence: ../feature-family-audits/20-numeric-feature-family-audit.ipynb


## 8. Categorical feature template

Use the same routine for each categorical feature:

1. count distinct values, nulls and the most common levels;
2. calculate how much of the column the largest categories cover;
3. inspect rare levels and labels such as `none` or `unknown`;
4. compare target proportions within the most common levels;
5. check whether spelling, case or whitespace splits one category into several;
6. record whether to retain, group or exclude the feature in the baseline.

Limit plots to the most common levels. A forty-category legend is decorative fog.

In [9]:
categorical_rows = feature_register["review type"].str.contains(
    "categorical|hierarchy|related",
    case=False,
    regex=True,
)
display(feature_register.loc[categorical_rows])
print("Focused notebooks use a shared 50-row rarity diagnostic and support-aware target tables.")


,feature,review type,focused audit,audit status,key finding,baseline treatment
9,basin,geographic categorical,../feature-family-audits/23-geographic-categor...,complete,Nine complete levels; train/test total-variati...,Retain as a stable low-cardinality category.
10,subvillage,geographic categorical,../feature-family-audits/23-geographic-categor...,complete,"19,287 levels; 87.32% of rows are in <50 group...",Exclude from first one-hot baseline; validate ...
11,region,geographic categorical,../feature-family-audits/23-geographic-categor...,complete,Twenty-one complete levels; functional rates r...,Retain as an interpretable low-cardinality bac...
12,region_code,geographic categorical,../feature-family-audits/23-geographic-categor...,complete,Twenty-seven categorical codes; region and cod...,Cast to category and compare with named region.
13,district_code,geographic categorical,../feature-family-audits/23-geographic-categor...,complete,Twenty reused numeric labels; code zero is not...,Cast to category and combine with region_code ...
14,lga,geographic categorical,../feature-family-audits/23-geographic-categor...,complete,"125 complete levels, no unseen test levels, an...",Retain; compare its signal against region back...
15,ward,geographic categorical,../feature-family-audits/23-geographic-categor...,complete,"2,092 levels; raw unseen exposure is 0.07%, or...",Retain via LGA+ward with fold-fitted rare/unse...
16,funder,high-cardinality categorical,../feature-family-audits/24-organisation-and-n...,complete,Effective missingness is 7.48%; 1.71% of test ...,"Keep null/sentinel distinct, normalise conserv..."
17,installer,high-cardinality categorical,../feature-family-audits/24-organisation-and-n...,complete,Effective missingness is 7.54%; case normalisa...,Keep separately from funder; trim/casefold/whi...
18,wpt_name,high-cardinality categorical,../feature-family-audits/24-organisation-and-n...,complete,56.90% of test names are unseen; leave-one-out...,Exclude raw exact names from baseline; test cr...


Focused notebooks use a shared 50-row rarity diagnostic and support-aware target tables.


## 9. Binary feature template

Treat `public_meeting` and `permit` as binary fields with possible missing values.

Check value counts, nulls and class proportions. Missing and `False` mean different things, however convenient merging them might be.

In [10]:
display(feature_register.loc[feature_register["review type"].eq("binary")])
print("public_meeting and permit keep true, false and source-blank states distinct.")


,feature,review type,focused audit,audit status,key finding,baseline treatment
7,public_meeting,binary,../feature-family-audits/22-binary-feature-fam...,complete,"The source is blank for 3,334 training rows (5...",Retain true/false/unknown as three explicit st...
8,permit,binary,../feature-family-audits/22-binary-feature-fam...,complete,"The source is blank for 3,056 training rows (5...",Retain true/false/unknown as three explicit st...


public_meeting and permit keep true, false and source-blank states distinct.


## 10. Date and construction-year template

Parse `date_recorded` and check the valid range, failed parses and number of records over time. Give `construction_year = 0` the suspicion it deserves.

A derived pump-age feature seems plausible. Check for negative ages and settle the unknown-year rule before using it.

In [11]:
display(feature_register.loc[feature_register["review type"].eq("temporal")])
print("All dates parse. Pump age is missing for unknown years and invalid for nine negative training cases.")


,feature,review type,focused audit,audit status,key finding,baseline treatment
5,date_recorded,temporal,../feature-family-audits/21-temporal-feature-f...,complete,All dates parse; training spans 2002-10-14 to ...,Derive recording year/month and elapsed days; ...
6,construction_year,temporal,../feature-family-audits/21-temporal-feature-f...,complete,Year zero affects 34.86%; valid pump age has m...,Keep unknown/inconsistent flags and valid pump...


All dates parse. Pump age is missing for unknown years and invalid for nine negative training cases.


## 11. Identifier and free-text template

Measure uniqueness, repeated values and missingness for `id` and `wpt_name`. Decide whether each column describes the pump or identifies the row.

A row ID is not a useful feature merely because the model accepts numbers. Skip target charts with thousands of labels.

In [12]:
display(
    feature_register.loc[
        feature_register["review type"].eq("high-cardinality categorical")
    ]
)
print("Raw exact waterpoint names are excluded from the first baseline because 56.90% of test rows are unseen.")


,feature,review type,focused audit,audit status,key finding,baseline treatment
16,funder,high-cardinality categorical,../feature-family-audits/24-organisation-and-n...,complete,Effective missingness is 7.48%; 1.71% of test ...,"Keep null/sentinel distinct, normalise conserv..."
17,installer,high-cardinality categorical,../feature-family-audits/24-organisation-and-n...,complete,Effective missingness is 7.54%; case normalisa...,Keep separately from funder; trim/casefold/whi...
18,wpt_name,high-cardinality categorical,../feature-family-audits/24-organisation-and-n...,complete,56.90% of test names are unseen; leave-one-out...,Exclude raw exact names from baseline; test cr...


Raw exact waterpoint names are excluded from the first baseline because 56.90% of test rows are unseen.


## 12. Geographic review

Longitude and latitude belong together. Check their ranges, count `(0, 0)` coordinates and plot the non-zero locations. Colouring by target may expose regional structure; it cannot explain it.

Compare the coordinates with `region`, `lga`, `ward` and the code fields. A random split may place neighbouring pumps on both sides, so note the validation risk.

In [13]:
display(
    feature_register.loc[
        feature_register["review type"].str.contains("geographic")
    ]
)
print("Regional functional rates span 29.75% to 78.22%; add an LGA/region-grouped robustness check.")


,feature,review type,focused audit,audit status,key finding,baseline treatment
1,longitude + latitude,paired geographic,20-longitude-latitude-pair-deep-dive.ipynb,complete,"The pair (0, approximately 0) encodes 1,812 mi...",Keep availability; replace the paired sentinel...
9,basin,geographic categorical,../feature-family-audits/23-geographic-categor...,complete,Nine complete levels; train/test total-variati...,Retain as a stable low-cardinality category.
10,subvillage,geographic categorical,../feature-family-audits/23-geographic-categor...,complete,"19,287 levels; 87.32% of rows are in <50 group...",Exclude from first one-hot baseline; validate ...
11,region,geographic categorical,../feature-family-audits/23-geographic-categor...,complete,Twenty-one complete levels; functional rates r...,Retain as an interpretable low-cardinality bac...
12,region_code,geographic categorical,../feature-family-audits/23-geographic-categor...,complete,Twenty-seven categorical codes; region and cod...,Cast to category and compare with named region.
13,district_code,geographic categorical,../feature-family-audits/23-geographic-categor...,complete,Twenty reused numeric labels; code zero is not...,Cast to category and combine with region_code ...
14,lga,geographic categorical,../feature-family-audits/23-geographic-categor...,complete,"125 complete levels, no unseen test levels, an...",Retain; compare its signal against region back...
15,ward,geographic categorical,../feature-family-audits/23-geographic-categor...,complete,"2,092 levels; raw unseen exposure is 0.07%, or...",Retain via LGA+ward with fold-fitted rare/unse...


Regional functional rates span 29.75% to 78.22%; add an LGA/region-grouped robustness check.


## 13. Related categorical hierarchies

Check whether each detailed value maps to one parent value in these families:

- `extraction_type` → `extraction_type_group` → `extraction_type_class`
- `management` → `management_group`
- `payment` → `payment_type`
- `water_quality` → `quality_group`
- `quantity` → `quantity_group`
- `source` → `source_type` → `source_class`
- `waterpoint_type` → `waterpoint_type_group`

Record inconsistencies and cardinality at each level. Start the baseline with one defensible level per concept. The entire hierarchy has to earn its place.

In [14]:
hierarchy_rows = feature_register["review type"].isin([
    "categorical hierarchy",
    "related categorical",
    "low-cardinality categorical",
])
display(feature_register.loc[hierarchy_rows])
print("Documented extraction, source, quality, management and waterpoint child-parent mappings are deterministic; scheme_name to scheme_management is not.")


,feature,review type,focused audit,audit status,key finding,baseline treatment
19,scheme_management,related categorical,../feature-family-audits/25-management-feature...,complete,"3,877 training rows are blank; it overlaps man...",Retain unknown as a level; compare against man...
20,scheme_name,related categorical,../feature-family-audits/25-management-feature...,complete,"28,166 rows are blank, 644 literal None; 1.26%...",Omit raw one-hot form initially; preserve blan...
21,management,related categorical,../feature-family-audits/25-management-feature...,complete,Twelve complete levels map deterministically t...,Use as the initial granular management represe...
22,management_group,related categorical,../feature-family-audits/25-management-feature...,complete,Deterministic coarse parent retains much less ...,Treat as likely redundant; compare coarse vers...
23,extraction_type,categorical hierarchy,../feature-family-audits/26-infrastructure-fea...,complete,Eighteen levels map deterministically upward; ...,Keep the granular type; rare-pool if validatio...
24,extraction_type_group,categorical hierarchy,../feature-family-audits/26-infrastructure-fea...,complete,Deterministic from type and only 0.0013 descri...,Treat as a strong redundancy candidate beside ...
25,extraction_type_class,categorical hierarchy,../feature-family-audits/26-infrastructure-fea...,complete,Seven deterministic broad classes give a compa...,Compare type plus class against type alone; do...
26,source,categorical hierarchy,../feature-family-audits/26-infrastructure-fea...,complete,Ten fully test-covered levels retain lake/rive...,Keep granular source and compare with the inte...
27,source_type,categorical hierarchy,../feature-family-audits/26-infrastructure-fea...,complete,Deterministic from source and removes useful w...,Use only as a lower-cardinality ablation candi...
28,source_class,categorical hierarchy,../feature-family-audits/26-infrastructure-fea...,complete,Three deterministic classes have weak descript...,First source-hierarchy level to omit from base...


Documented extraction, source, quality, management and waterpoint child-parent mappings are deterministic; scheme_name to scheme_management is not.


## 14. Consolidate the missing-data audit

Bring the feature reviews together and separate:

- explicit nulls recognised by the DataFrame;
- numeric sentinels such as zero where zero is implausible;
- categorical sentinels such as `none` or `unknown`;
- genuine zero and `False` values.

Use one proposed rule per affected column. No grand replace-all-zeros manoeuvre.

In [15]:
missing_data_register = pd.DataFrame([
    {"feature/state": "amount_tsh = 0", "training rows": 41639, "test rows": 10410, "treatment": "availability plus transformed positive magnitude"},
    {"feature/state": "coordinate pair sentinel", "training rows": 1812, "test rows": 457, "treatment": "paired availability then missing coordinates"},
    {"feature/state": "gps_height = 0", "training rows": 20438, "test rows": 5211, "treatment": "availability then fold-fitted imputation"},
    {"feature/state": "population = 0", "training rows": 21381, "test rows": 5453, "treatment": "keep separate from population=1"},
    {"feature/state": "construction_year = 0", "training rows": 20709, "test rows": 5260, "treatment": "unknown-year and valid-age features"},
    {"feature/state": "public_meeting blank", "training rows": 3334, "test rows": 821, "treatment": "third categorical state"},
    {"feature/state": "permit blank", "training rows": 3056, "test rows": 737, "treatment": "third categorical state"},
    {"feature/state": "funder blank/sentinel", "training rows": 4445, "test rows": 1079, "treatment": "keep blank and sentinel distinct"},
    {"feature/state": "installer blank/sentinel", "training rows": 4481, "test rows": 1087, "treatment": "keep blank and sentinel distinct"},
    {"feature/state": "scheme_name blank", "training rows": 28166, "test rows": 7092, "treatment": "distinct from literal None/none"},
])
missing_data_register["training (%)"] = (
    missing_data_register["training rows"] / len(training_set_values) * 100
).round(2)
missing_data_register["test (%)"] = (
    missing_data_register["test rows"] / len(test_set_values) * 100
).round(2)
display(missing_data_register)


,feature/state,training rows,test rows,treatment,training (%),test (%)
0,amount_tsh = 0,41639,10410,availability plus transformed positive magnitude,70.10,70.10
1,coordinate pair sentinel,1812,457,paired availability then missing coordinates,3.05,3.08
2,gps_height = 0,20438,5211,availability then fold-fitted imputation,34.41,35.09
3,population = 0,21381,5453,keep separate from population=1,35.99,36.72
4,construction_year = 0,20709,5260,unknown-year and valid-age features,34.86,35.42
5,public_meeting blank,3334,821,third categorical state,5.61,5.53
6,permit blank,3056,737,third categorical state,5.14,4.96
7,funder blank/sentinel,4445,1079,keep blank and sentinel distinct,7.48,7.27
8,installer blank/sentinel,4481,1087,keep blank and sentinel distinct,7.54,7.32
9,scheme_name blank,28166,7092,distinct from literal None/none,47.42,47.76


## 15. Compare training and test predictors

Compare shape, missingness, numeric ranges and category levels. Count categories that appear in the test set but not the training set.

Use the result to design the encoder and pipeline. The test file has no target and gets no vote on model performance.

In [16]:
predictor_drift_summary = pd.DataFrame([
    {"family": "numeric", "evidence": "maximum full-distribution KS D = 0.0112", "decision": "No material marginal shift; retain range and sentinel guards."},
    {"family": "temporal", "evidence": "recording-year/month total variation < 0.9 percentage points", "decision": "Use training-defined date derivations."},
    {"family": "binary", "evidence": "total variation <= 0.28 percentage points", "decision": "Keep explicit unknown handling."},
    {"family": "low-cardinality categories", "evidence": "supplied test levels are broadly covered", "decision": "Still configure handle_unknown for validation/future data."},
    {"family": "subvillage", "evidence": "16.09% of test rows use unseen levels", "decision": "Exclude raw one-hot form from baseline."},
    {"family": "wpt_name", "evidence": "56.90% of test rows use unseen exact names", "decision": "Exclude raw exact names from baseline."},
    {"family": "scheme_name", "evidence": "about 1.1% unseen after conservative normalisation", "decision": "Use only a fold-fitted high-cardinality experiment."},
])
display(predictor_drift_summary)


,family,evidence,decision
0,numeric,maximum full-distribution KS D = 0.0112,No material marginal shift; retain range and s...
1,temporal,recording-year/month total variation < 0.9 per...,Use training-defined date derivations.
2,binary,total variation <= 0.28 percentage points,Keep explicit unknown handling.
3,low-cardinality categories,supplied test levels are broadly covered,Still configure handle_unknown for validation/...
4,subvillage,16.09% of test rows use unseen levels,Exclude raw one-hot form from baseline.
5,wpt_name,56.90% of test rows use unseen exact names,Exclude raw exact names from baseline.
6,scheme_name,about 1.1% unseen after conservative normalisa...,Use only a fold-fitted high-cardinality experi...


## 16. Check duplicates and consistency

Look for duplicate rows with and without `id`, impossible values, conflicting category mappings and columns with one value.

Classify each result as a data error, a legitimate repeated observation or a feature with no predictive information.

In [17]:
consistency_summary = pd.DataFrame([
    {"relationship": "quantity_group / quantity", "finding": "exact duplicate", "action": "drop quantity_group"},
    {"relationship": "payment / payment_type", "finding": "fixed relabelling", "action": "drop payment"},
    {"relationship": "recorded_by", "finding": "constant", "action": "drop recorded_by"},
    {"relationship": "management / management_group", "finding": "deterministic child-to-parent", "action": "prefer management; validate coarse ablation"},
    {"relationship": "extraction/source/quality/waterpoint families", "finding": "deterministic documented hierarchies", "action": "do not keep every level automatically"},
    {"relationship": "lga / region", "finding": "LGA maps deterministically to region", "action": "retain region as low-cardinality back-off"},
    {"relationship": "region / region_code", "finding": "not simple duplicates", "action": "treat code categorically and compare"},
    {"relationship": "scheme_name / scheme_management", "finding": "non-deterministic", "action": "do not derive one mechanically from the other"},
])
display(consistency_summary)


,relationship,finding,action
0,quantity_group / quantity,exact duplicate,drop quantity_group
1,payment / payment_type,fixed relabelling,drop payment
2,recorded_by,constant,drop recorded_by
3,management / management_group,deterministic child-to-parent,prefer management; validate coarse ablation
4,extraction/source/quality/waterpoint families,deterministic documented hierarchies,do not keep every level automatically
5,lga / region,LGA maps deterministically to region,retain region as low-cardinality back-off
6,region / region_code,not simple duplicates,treat code categorically and compare
7,scheme_name / scheme_management,non-deterministic,do not derive one mechanically from the other


## 17. Apply the settled structural column removals

The settled duplicate and constant-column evidence, recorded in [`data-preparation-next-steps.md`](../reports/data-preparation-next-steps.md), supports three fixed removals: `quantity_group`, `payment` and `recorded_by`. Apply the shared function independently to the two raw feature DataFrames.

This is deliberately not imputation, learned feature selection or dimensionality reduction. Keep `id`, preserve the raw source frames and give the returned frames names that describe exactly what has happened.

In [18]:
# Apply only the settled, non-statistical column removals.
from data_preparation import remove_known_redundant_columns

training_features_after_column_removal = (
    remove_known_redundant_columns(training_set_values)
)
test_features_after_column_removal = (
    remove_known_redundant_columns(test_set_values)
)

training_removed_columns = [
    column
    for column in training_set_values.columns
    if column not in training_features_after_column_removal.columns
]
test_removed_columns = [
    column
    for column in test_set_values.columns
    if column not in test_features_after_column_removal.columns
]

if training_removed_columns != test_removed_columns:
    raise ValueError(
        "Training and test data had different structural removals."
    )

column_removal_summary = pd.DataFrame([
    {
        "dataset": "Training",
        "source columns": training_set_values.shape[1],
        "returned columns": training_features_after_column_removal.shape[1],
        "id retained": "id" in training_features_after_column_removal,
        "removed columns": ", ".join(training_removed_columns),
    },
    {
        "dataset": "Test",
        "source columns": test_set_values.shape[1],
        "returned columns": test_features_after_column_removal.shape[1],
        "id retained": "id" in test_features_after_column_removal,
        "removed columns": ", ".join(test_removed_columns),
    },
]).set_index("dataset")

print(f"Removed columns: {training_removed_columns}")
column_removal_summary


Removed columns: ['recorded_by', 'payment', 'quantity_group']


,source columns,returned columns,id retained,removed columns
dataset,,,,
Training,40,37,True,"recorded_by, payment, quantity_group"
Test,40,37,True,"recorded_by, payment, quantity_group"


## 18. Consolidated predictor findings and decision log

The detailed evidence lives in the focused notebooks. This section keeps
the cross-cutting findings needed by notebook 02 in one place.


In [19]:
audit_decision_summary = pd.DataFrame([
    {"finding": "candidate predictor coverage", "evidence": "36 of 36", "action": "Proceed to a fixed split and preprocessing baseline."},
    {"finding": "settled raw-column removals", "evidence": "quantity_group, payment, recorded_by", "action": "Apply the shared fixed removal to training and test."},
    {"finding": "source-token preservation", "evidence": "28,166 blank scheme names versus 644 literal None values", "action": "Load with keep_default_na=False; distinguish blanks from semantic sentinels."},
    {"finding": "shared zero-measurement block", "evidence": "19,668 training rows (33.11%) have zero height, population and construction year", "action": "Create feature-specific flags; never replace all numeric zeros globally."},
    {"finding": "population special states", "evidence": "population=1 covers 7,025 rows and is 53.52% non-functional", "action": "Keep zero and one distinct from ordinary positive magnitude."},
    {"finding": "nullable binary fields", "evidence": "public_meeting blank 5.61%; permit blank 5.15%", "action": "Preserve unknown as a third state independently for each field."},
    {"finding": "temporal derivation", "evidence": "valid pump age median 13; non-functional rises from 23.73% at 1-5 years to 67.24% at 41+", "action": "Derive valid age and inconsistency/unknown flags; fit imputation inside folds."},
    {"finding": "sparse location/name fields", "evidence": "unseen test rows: subvillage 16.09%, wpt_name 56.90%", "action": "Exclude raw one-hot forms from baseline; test fold-safe hashing/frequency approaches separately."},
    {"finding": "waterpoint-name optimism", "evidence": "84.70% in-sample category-mode accuracy falls to 55.01% leave-one-out lookup", "action": "Do not use raw exact wpt_name in the first baseline."},
    {"finding": "scheme-name coverage", "evidence": "47.42% source blank; unseen test names affect 1.26%", "action": "Keep blank/None distinct and test only fold-fitted high-cardinality treatments."},
    {"finding": "deterministic hierarchies", "evidence": "extraction, source, quality, management and waterpoint child-parent mappings", "action": "Prefer a validated granular or coarse level; do not automatically retain all levels."},
    {"finding": "strong service-state associations", "evidence": "quantity=dry is 96.89% non-functional; water_quality=unknown is 84.06%", "action": "Retain as nominal categories with explicit unknown and support-aware interpretation."},
    {"finding": "geographic dependence", "evidence": "regional functional rate spans 29.75% to 78.22%", "action": "Keep the stratified split and add an LGA/region-grouped robustness check."},
    {"finding": "aggregate train/test stability", "evidence": "small numeric KS, binary TV and broad categorical marginal differences", "action": "Still configure unseen handling; absence of current drift does not guarantee future coverage."},
])
display(audit_decision_summary)


,finding,evidence,action
0,candidate predictor coverage,36 of 36,Proceed to a fixed split and preprocessing bas...
1,settled raw-column removals,"quantity_group, payment, recorded_by",Apply the shared fixed removal to training and...
2,source-token preservation,"28,166 blank scheme names versus 644 literal N...",Load with keep_default_na=False; distinguish b...
3,shared zero-measurement block,"19,668 training rows (33.11%) have zero height...",Create feature-specific flags; never replace a...
4,population special states,"population=1 covers 7,025 rows and is 53.52% n...",Keep zero and one distinct from ordinary posit...
5,nullable binary fields,public_meeting blank 5.61%; permit blank 5.15%,Preserve unknown as a third state independentl...
6,temporal derivation,valid pump age median 13; non-functional rises...,Derive valid age and inconsistency/unknown fla...
7,sparse location/name fields,"unseen test rows: subvillage 16.09%, wpt_name ...",Exclude raw one-hot forms from baseline; test ...
8,waterpoint-name optimism,84.70% in-sample category-mode accuracy falls ...,Do not use raw exact wpt_name in the first bas...
9,scheme-name coverage,47.42% source blank; unseen test names affect ...,Keep blank/None distinct and test only fold-fi...


## 19. Bridge to the taught modelling workflow

Keep the course sequence visible in notebook 02:

1. choose the validation measure and create a reproducible stratified split;
2. establish a majority-class reference and an interpretable decision-tree baseline;
3. place imputation and encoding inside the training pipeline;
4. test oversampling on training data only;
5. use k-fold cross-validation for model comparison;
6. tune the strongest taught model after the baseline works.

Methods beyond the course can wait until this sequence works. Each addition needs a problem from the audit and a repeatable validation improvement; novelty alone is not a result.

## Completion checklist

- [x] Source files and identifier alignment are validated.
- [x] The target has a separate integrity and class-balance audit.
- [x] Each of the 36 candidate predictors has an entry in the feature register.
- [x] Explicit missing values and suspected sentinels are separated.
- [x] Related categorical and geographic features have paired consistency checks.
- [x] Training/test ranges, levels and unseen-category exposure are compared.
- [x] The fixed structural removal returns 37 columns while retaining `id`.
- [x] Every proposed baseline treatment points to a focused audit finding.
- [ ] Recheck target relationships after the stratified development/validation split is frozen.
- [ ] Compare high-cardinality, hierarchy and geographic choices by held-out validation.
